# Заргат — генерация 3D-модели через Hunyuan3D-2mini

Подготовленный запуск для Google Colab. Эта версия сначала создаёт **геометрию без текстуры**: так выше шанс уложиться в доступную бесплатную видеопамять. Риггинг и игровые анимации будут отдельным этапом.

В Colab выбери GPU в меню среды выполнения. Бесплатный GPU выдаётся не всегда. Не покупай подписку ради этого теста.

In [ ]:
import os, torch, platform\nprint('Python:', platform.python_version())\nprint('PyTorch:', torch.__version__)\nif not torch.cuda.is_available():\n    raise RuntimeError('GPU не выдана. Открой Среда выполнения → Сменить среду выполнения → GPU. Если GPU недоступна бесплатно, остановись здесь.')\nprint('GPU:', torch.cuda.get_device_name(0))\nprint('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 1. Установка официального Hunyuan3D-2

Клонируем исходный код Tencent непосредственно из официального репозитория.

In [ ]:
%cd /content\n!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git\n%cd /content/Hunyuan3D-2\n!pip install -r requirements.txt\n!pip install -e .\n%cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer\n!python setup.py install\n%cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer\n!python setup.py install\n%cd /content/Hunyuan3D-2

## 2. Загрузи концепт Заргата

Для лучшего результата нужен один фронтальный вид персонажа целиком, без подписей и дополнительных панелей. Если у тебя только общий лист-концепт, сначала подготовь отдельный фронтальный кадр. Не загружай весь лист с несколькими ракурсами в один входной кадр.

In [ ]:
from google.colab import files\nfrom PIL import Image\nuploaded = files.upload()\nif not uploaded:\n    raise RuntimeError('Изображение не загружено')\ninput_image_path = '/content/' + next(iter(uploaded.keys()))\nsource_image = Image.open(input_image_path).convert('RGB')\ndisplay(source_image.resize((512, 512)))

## 3. Генерация формы (Hunyuan3D-2mini)

Используется компактная модель 0.6B. Она создаёт меш по одному изображению, но не гарантирует точное совпадение всех деталей и не создаёт скелет.

In [ ]:
import torch\nfrom hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline\nshape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(\n    'tencent/Hunyuan3D-2mini',\n    subfolder='hunyuan3d-dit-v2-mini',\n    use_safetensors=True,\n    device='cuda'\n)\nwith torch.inference_mode():\n    result = shape_pipe(\n        image=input_image_path,\n        num_inference_steps=30,\n        octree_resolution=380,\n        num_chunks=20000,\n        generator=torch.manual_seed(12345),\n        output_type='trimesh'\n    )\nmesh = result[0]\noutput_path = '/content/zargat_shape.glb'\nmesh.export(output_path)\nprint('Создано:', output_path)\nprint('Размер MB:', round(os.path.getsize(output_path) / 1024**2, 2))

## 4. Скачать модель

Скачай GLB и передай его в проект. Следом мы проверим форму в Blender, уберём лишнюю геометрию и подготовим к скелету и анимациям. Текстуры добавим после проверки формы: полный этап shape + texture требует существенно больше VRAM.

In [ ]:
files.download(output_path)